In [35]:
import torch
from torchmetrics.functional.image import image_gradients

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import xarray as xr
import matplotlib.pyplot as plt
import math
from plotly.subplots import make_subplots

torch.set_printoptions(sci_mode = False)

In [2]:
t = torch.load('./torch_data/ea_scenes_tensor.pt')

In [32]:
t_bed = t[:, 0].unsqueeze(1) # N, C, H, W format with C = 1
t_sur = t[:, 1].unsqueeze(1) # N, C, H, W format with C = 1

In [63]:
t[40, 7, :, :] # 6 is the y channel, 7 the x channel
# x increases from left to right
# y top row (first row) has the highest y values

tensor([389.5017, 388.7056, 388.0154, 387.5264, 387.0505, 386.9680, 387.4124,
        387.5911, 388.1650, 389.1208, 390.1614, 391.0776, 392.3208, 393.5845,
        394.5696, 395.8210, 397.1543, 398.3860, 399.7168, 401.1597, 402.4248,
        403.8652, 405.2505, 406.4023, 407.8406, 409.2886, 411.0857, 412.8948,
        415.3569, 418.0012, 421.1211, 424.3171, 427.8794, 431.6860, 435.5261,
        439.9373, 444.1506, 449.1042, 454.1492, 459.1707, 463.9233, 468.9731,
        474.6326, 480.2395, 486.8318, 493.2512, 499.7161, 506.1379, 512.7356,
        519.6394, 527.0781, 534.0667, 541.3059, 548.7595, 555.7356, 562.3394,
        568.7542, 574.6506, 579.9329, 584.8438])

In [75]:
bed_dy, bed_dx = image_gradients(t_bed)
sur_dy, sur_dx = image_gradients(t_sur)

In [65]:
fig = px.imshow(t_bed[40].squeeze(), 
                color_continuous_scale = 'gray',
                # y_indexing is opposite of coordinates but this does not matter
                title = "Bed topography")

fig.update_layout(font_family = "Times New Roman")
fig.update_yaxes(ticksuffix = "  ")

fig.show()

In [114]:
# need to transpose to get into right shape
# t_bed[40].squeeze().mT
fig = go.Figure(data = [go.Surface(z = t_bed[40].squeeze(), colorscale = "haline", lighting = dict(ambient = 0.9))])
fig.update_layout(template = "plotly_white") # simple_white

# fig.update_layout(height = 20)

fig.show()

In [102]:
fig = px.imshow(bed_dy[40].squeeze(), 
                color_continuous_scale = "RdBu_r", color_continuous_midpoint = 0,
                # y_indexing is opposite of coordinates but this does not matter
                title = "Bed topography dx")

fig.update_layout(font_family = "Times New Roman")
fig.update_yaxes(ticksuffix = "  ")

fig.show()

In [73]:
fig = px.imshow(bed_dx[40].squeeze(), 
                color_continuous_scale = "RdBu_r", color_continuous_midpoint = 0,
                # y_indexing is opposite of coordinates but this does not matter
                title = "Bed topography dx")

fig.update_layout(font_family = "Times New Roman")
fig.update_yaxes(ticksuffix = "  ")

fig.show()

https://plotly.com/python/builtin-colorscales/

Best sequential:
- Plasma
- Blackbody
- YlOrRd
- haline
Tealgrn

Best diverging:
- RdYlBu
- RdYlGn
- delta
- Temps

Best cyclical:
- IceFire

Other:
- winter

- dy: image[1,:] (second row) - image[0,:] (first row).  
- If positive: It is increasing in top to bottom direction.

- (H - 1) and (W - 1) dims respectively. Could inlude extra row as Antarctica does not stop there. Need n+1 for full gradient

- Change dimension
- Unit is meters.
- One reference pixel value to "anchor" the represntation would be enough to reconstruct.

# Corr

In [22]:
t_flat = t.permute(1, 0, 2, 3).reshape(8, -1)

In [23]:
torch.corrcoef(t_flat)
# 0.53 correlation between surface elevation and bed elevation

tensor([[     1.0000,      0.5311,     -0.7364,         nan,      0.4205,
             -0.2492,     -0.0725,     -0.5098],
        [     0.5311,      1.0000,      0.1821,         nan,      0.9580,
             -0.0084,     -0.5039,     -0.7774],
        [    -0.7364,      0.1821,      1.0000,         nan,      0.2768,
              0.2826,     -0.3182,     -0.0291],
        [        nan,         nan,         nan,         nan,         nan,
                 nan,         nan,         nan],
        [     0.4205,      0.9580,      0.2768,         nan,      1.0000,
              0.0506,     -0.5060,     -0.7990],
        [    -0.2492,     -0.0084,      0.2826,         nan,      0.0506,
              1.0000,      0.1264,     -0.0888],
        [    -0.0725,     -0.5039,     -0.3182,         nan,     -0.5059,
              0.1264,      1.0000,     -0.0000],
        [    -0.5098,     -0.7774,     -0.0291,         nan,     -0.7990,
             -0.0888,     -0.0000,      1.0000]])

In [26]:
t_flat[0]
t_flat[1]

tensor([3689.1274, 3689.2078, 3689.2009,  ..., 3578.1594, 3577.8826,
        3577.7095])

In [86]:
torch.corrcoef(
    torch.cat((
    bed_dy.permute(1, 0, 2, 3).reshape(1, -1),
    sur_dy.permute(1, 0, 2, 3).reshape(1, -1),
    bed_dx.permute(1, 0, 2, 3).reshape(1, -1),
    sur_dx.permute(1, 0, 2, 3).reshape(1, -1)), 
    dim = 0))

tensor([[ 1.0000,  0.0309,  0.0535,  0.0196],
        [ 0.0309,  1.0000, -0.0036, -0.2495],
        [ 0.0535, -0.0036,  1.0000,  0.0492],
        [ 0.0196, -0.2495,  0.0492,  1.0000]])

In [90]:
bed_dy.permute(1, 0, 2, 3)[:, 40, :, :].shape

torch.Size([1, 60, 60])

In [91]:
torch.corrcoef(
    torch.cat((
    bed_dy.permute(1, 0, 2, 3)[:, 40, :, :].reshape(1, -1),
    sur_dy.permute(1, 0, 2, 3)[:, 40, :, :].reshape(1, -1),
    bed_dx.permute(1, 0, 2, 3)[:, 40, :, :].reshape(1, -1),
    sur_dx.permute(1, 0, 2, 3)[:, 40, :, :].reshape(1, -1)), 
    dim = 0))

tensor([[ 1.0000, -0.0660,  0.0566, -0.0438],
        [-0.0660,  1.0000,  0.0103, -0.3695],
        [ 0.0566,  0.0103,  1.0000, -0.1545],
        [-0.0438, -0.3695, -0.1545,  1.0000]])

In this domain there actaully is very little relationship between the gradients.

Need per image mean
Normalisation issue? Different in diff domains?